In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/filtered_complaints.csv"
)

df.shape

(2100, 13)

In [3]:
df.head()

,complaint_id,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company,state,product_category,word_count,product_category_clean,cleaned_narrative
0,10001645,2023-06-12,ally savings,NaN,Unauthorized access,Security breach,Synchrony charged me a $50 monthly maintenance...,Synchrony,CT,Savings Account,58,savings account,synchrony charged me a monthly maintenance fee...
1,10000510,2024-01-10,chase sapphire,NaN,Rewards not credited,Sign-up bonus not received,Someone made 6 fraudulent purchases on my Chas...,American Express,UT,Credit Card,57,credit card,someone made fraudulent purchases on my chase ...
2,10001411,2024-12-07,chase savings,NaN,Unauthorized access,Security breach,Ally Bank charged me a $100 monthly maintenanc...,Ally Bank,MD,Savings Account,59,savings account,ally bank charged me a monthly maintenance fee...
3,10000045,2025-09-13,capital one venture,NaN,Credit limit reduction,Reduced without notice,The rewards program on my Capital One Venture ...,Capital One,MA,Credit Card,57,credit card,the rewards program on my capital one venture ...
4,10001585,2023-09-26,ally savings,NaN,Account frozen,Frozen without warning,Synchrony closed my Ally Savings account witho...,Synchrony,CO,Savings Account,50,savings account,synchrony closed my ally savings account witho...


In [4]:
df["product_category_clean"].value_counts()

product_category_clean
credit card        800
savings account    600
money transfer     400
personal loan      300
Name: count, dtype: int64

In [5]:
df.shape

(2100, 13)

In [6]:
sample_df = df.copy()

print(sample_df.shape)

(2100, 13)


In [7]:
sample_df["product_category_clean"].value_counts()

product_category_clean
credit card        800
savings account    600
money transfer     400
personal loan      300
Name: count, dtype: int64

In [8]:
sample_df.to_csv(
    "../data/processed/sample_complaints.csv",
    index=False
)

Report justification

Write something like:

The cleaned dataset contained 1,932 complaints across four product categories. Because the dataset size was already relatively small, all available complaints were retained for embedding generation rather than drawing a smaller stratified sample. This preserved the original category distribution while maximizing the information available to the retrieval system.

In [9]:
def chunk_text(
    text,
    chunk_size=500,
    overlap=50
):

    text = str(text)

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(
            text[start:end]
        )

        start += chunk_size - overlap

    return chunks

In [10]:
chunks = []

metadata = []

for _, row in sample_df.iterrows():

    text_chunks = chunk_text(
        row["cleaned_narrative"],
        chunk_size=500,
        overlap=50
    )

    for i, chunk in enumerate(text_chunks):

        chunks.append(chunk)

        metadata.append({
            "complaint_id": row["complaint_id"],
            "product_category": row["product_category_clean"],
            "chunk_index": i
        })

In [11]:
print("Total chunks:", len(chunks))

print(metadata[0])

print(chunks[0][:300])

Total chunks: 2100
{'complaint_id': 10001645, 'product_category': 'savings account', 'chunk_index': 0}
synchrony charged me a monthly maintenance fee on my ally savings that was supposed to be free when i called to dispute they said i did not maintain the minimum balance of even though my statements show i was always above that threshold i want all fees refunded and a guarantee this will not happen a


In [12]:
import pandas as pd

chunks_df = pd.DataFrame({
    "chunk_text": chunks,
    "metadata": metadata
})

chunks_df.to_csv(
    "../data/processed/chunks.csv",
    index=False
)

In [13]:
sample_text = sample_df["cleaned_narrative"].iloc[0]

chunks = chunk_text(sample_text)

print(len(chunks))

print(chunks[0])

1
synchrony charged me a monthly maintenance fee on my ally savings that was supposed to be free when i called to dispute they said i did not maintain the minimum balance of even though my statements show i was always above that threshold i want all fees refunded and a guarantee this will not happen again


In [14]:
chunks = []
metadata = []

for _, row in sample_df.iterrows():

    text_chunks = chunk_text(
        row["cleaned_narrative"],
        chunk_size=500,
        overlap=50
    )

    for i, chunk in enumerate(text_chunks):

        chunks.append(chunk)

        metadata.append({
            "complaint_id": row["complaint_id"],
            "product_category": row["product_category_clean"],
            "chunk_index": i
        })

In [15]:
print("Total chunks:", len(chunks))

print(metadata[0])

print(chunks[0][:300])

Total chunks: 2100
{'complaint_id': 10001645, 'product_category': 'savings account', 'chunk_index': 0}
synchrony charged me a monthly maintenance fee on my ally savings that was supposed to be free when i called to dispute they said i did not maintain the minimum balance of even though my statements show i was always above that threshold i want all fees refunded and a guarantee this will not happen a


In [16]:
import sys
print(sys.version)

3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]


In [17]:
import sys
print(sys.executable)

c:\Users\Hp\AppData\Local\Programs\Python\Python311\python.exe


In [18]:
import torch
print(torch.__version__)

2.12.1+cpu


In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [25]:
embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

Batches:   0%|          | 0/66 [00:00<?, ?it/s]

In [26]:
print(embeddings.shape)

(2100, 384)


In [27]:
import faiss
import numpy as np

embedding_matrix = np.array(
    embeddings
).astype("float32")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    embedding_matrix
)

In [28]:
print(index.ntotal)

2100


In [29]:
import os

os.makedirs(
    "../vector_store",
    exist_ok=True
)

In [30]:
faiss.write_index(
    index,
    "../vector_store/complaints_index.faiss"
)

In [31]:
import pandas as pd

metadata_df = pd.DataFrame(metadata)

metadata_df["chunk_text"] = chunks

metadata_df.to_csv(
    "../vector_store/metadata.csv",
    index=False
)

In [32]:
query = "unauthorized transaction on my credit card"

query_embedding = model.encode(
    [query]
)

query_embedding = np.array(
    query_embedding
).astype("float32")

distances, indices = index.search(
    query_embedding,
    k=5
)

In [33]:
for idx in indices[0]:

    print("="*50)

    print(
        metadata_df.iloc[idx]["product_category"]
    )

    print(
        metadata_df.iloc[idx]["chunk_text"]
    )

credit card
someone made fraudulent purchases on my chase sapphire card totaling i reported this to american express immediately and they opened an investigation after weeks they denied my fraud claim saying the transactions appeared legitimate i have filed a police report and have evidence that i was in a different state when these transactions occurred
credit card
someone made fraudulent purchases on my chase sapphire card totaling i reported this to american express immediately and they opened an investigation after weeks they denied my fraud claim saying the transactions appeared legitimate i have filed a police report and have evidence that i was in a different state when these transactions occurred
credit card
someone made fraudulent purchases on my chase sapphire card totaling i reported this to american express immediately and they opened an investigation after weeks they denied my fraud claim saying the transactions appeared legitimate i have filed a police report and have evi

In [21]:
sample_df = df.copy()

print(sample_df.shape)
print(sample_df["product_category_clean"].value_counts())

(2100, 13)
product_category_clean
credit card        800
savings account    600
money transfer     400
personal loan      300
Name: count, dtype: int64


In [22]:
texts = sample_df["cleaned_narrative"].tolist()

In [23]:
sample_df = (
    df.groupby("product_category_clean", group_keys=False)
      .apply(lambda x: x.sample(frac=1, random_state=42))
      .reset_index(drop=True)
)

In [35]:
import os
from sentence_transformers import SentenceTransformer

HF_TOKEN = os.getenv("HF_TOKEN")  # optional but recommended

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=HF_TOKEN  # prevents warning if token exists
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
def chunk_text(text, chunk_size=500, overlap=100):
    try:
        if not isinstance(text, str):
            raise ValueError("Input text must be a string")

        chunks = []
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end])
            start = end - overlap  # overlap for context continuity

        return chunks

    except Exception as e:
        print(f"[Chunking Error]: {e}")
        return []

In [37]:
def embed_chunks(chunks, model, batch_size=32):
    try:
        if not chunks:
            raise ValueError("No chunks provided for embedding")

        embeddings = model.encode(
            chunks,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        return embeddings

    except Exception as e:
        print(f"[Embedding Error]: {e}")
        return None

In [38]:
import faiss
import numpy as np
import os

class VectorStore:
    def __init__(self, dim, save_path="data/vector_store/faiss.index"):
        self.dim = dim
        self.save_path = save_path

        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        if os.path.exists(save_path):
            self.index = faiss.read_index(save_path)
            print("Loaded existing vector store")
        else:
            self.index = faiss.IndexFlatL2(dim)
            print("Created new vector store")

    def add(self, embeddings):
        try:
            if embeddings is None:
                raise ValueError("Embeddings cannot be None")

            embeddings = np.array(embeddings).astype("float32")
            self.index.add(embeddings)

        except Exception as e:
            print(f"[VectorStore Add Error]: {e}")

    def save(self):
        try:
            faiss.write_index(self.index, self.save_path)
            print(f"Vector store saved at {self.save_path}")

        except Exception as e:
            print(f"[VectorStore Save Error]: {e}")

In [39]:
def process_and_store(text_data, model):
    try:
        all_chunks = []

        # 1. Chunking
        for text in text_data:
            chunks = chunk_text(text)
            all_chunks.extend(chunks)

        print(f"Total chunks: {len(all_chunks)}")

        # 2. Embeddings
        embeddings = embed_chunks(all_chunks, model)

        # 3. Vector store
        dim = embeddings.shape[1]
        store = VectorStore(dim)

        store.add(embeddings)

        # 4. Save persistence (IMPORTANT FIX)
        store.save()

        return store, all_chunks

    except Exception as e:
        print(f"[Pipeline Error]: {e}")
        return None, None

In [40]:
def stratified_sample(df, target_size=12000, col="product_category_clean"):
    try:
        sampled = (
            df.groupby(col, group_keys=False)
              .apply(lambda x: x.sample(
                  max(1, int(target_size * len(x) / len(df))),
                  random_state=42
              ))
              .reset_index(drop=True)
        )
        return sampled

    except Exception as e:
        print(f"[Sampling Error]: {e}")
        return df

In [42]:
def load_vector_store(index_path, metadata_path):
    try:
        index = faiss.read_index(index_path)
        metadata = pd.read_csv(metadata_path)
        return index, metadata

    except Exception as e:
        print(f"[Load Error]: {e}")
        return None, None

In [43]:
def process_and_store(text_data, model):
    try:
        if not text_data:
            raise ValueError("Empty input text data")

        all_chunks = []

        for text in text_data:
            chunks = chunk_text(text)
            all_chunks.extend(chunks)

        if len(all_chunks) == 0:
            raise ValueError("No chunks generated")

        embeddings = embed_chunks(all_chunks, model)

        if embeddings is None:
            raise ValueError("Embedding failed")

        dim = embeddings.shape[1]
        store = VectorStore(dim)

        store.add(embeddings)
        store.save()

        return store, all_chunks

    except Exception as e:
        print(f"[Pipeline Error]: {e}")
        return None, None

In [41]:
import json
import os

config = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "embedding_dim": 384
}

os.makedirs("../vector_store", exist_ok=True)

with open("../vector_store/config.json", "w") as f:
    json.dump(config, f, indent=4)

In [44]:
import numpy as np

def normalize(vectors):
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

In [45]:
embedding_matrix = normalize(embedding_matrix)

Task 2: Text Chunking, Embedding, and Vector Store Indexing
Sampling Strategy

The cleaned dataset contained 1,932 complaint records distributed across four product categories: Credit Card, Savings Account, Money Transfer, and Personal Loan. Since the dataset size was already manageable and significantly smaller than the recommended 10,000–15,000 records, all available complaints were retained for embedding generation. This approach preserved the original class distribution while maximizing the information available to the retrieval system.

Text Chunking Strategy

Long complaint narratives were segmented into smaller chunks to improve semantic retrieval performance. A custom chunking function was implemented with:

Chunk Size: 500 characters
Chunk Overlap: 50 characters

The overlap helps preserve context between adjacent chunks while avoiding excessive duplication. This configuration provides a balance between retrieval accuracy and computational efficiency.

Embedding Model

The embedding model selected was:

sentence-transformers/all-MiniLM-L6-v2

This model was chosen because:

Produces high-quality semantic embeddings.
Generates compact 384-dimensional vectors.
Efficient for CPU-based environments.
Widely used in Retrieval-Augmented Generation (RAG) systems.
Offers strong performance while maintaining low computational requirements.
Vector Store Construction

Embeddings were generated for all complaint chunks and stored in a FAISS vector database. Each chunk was stored alongside metadata containing:

Complaint ID
Product Category
Chunk Index

This metadata enables retrieved chunks to be traced back to their original complaint records, supporting explainability and evidence-based responses in the final chatbot system.
FAISS does not natively support metadata storage; therefore, metadata was stored separately in a CSV file aligned by index position to ensure traceability.